# Diabetic Retinopathy — Improved Model

**Improvements over baseline:**
| | Baseline | Improved |
|---|---|---|
| Backbone | 3-block CNN from scratch | EfficientNet-B0 (ImageNet pretrained) |
| Loss | CrossEntropy (uniform) | CrossEntropy + class weights + label smoothing |
| Augmentation | HFlip only | HFlip + VFlip + Rotation + ColorJitter + RandomCrop |
| Optimizer | Adam lr=1e-3 | AdamW lr=1e-4, wd=1e-4 |
| LR schedule | ReduceLROnPlateau | CosineAnnealingLR |
| Checkpoint | Best val accuracy | Best val macro F1 |
| Epochs | 20 | 30 |

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))

# ── CONFIG ─────────────────────────────────────────────────────────────────────
CSV_PATH   = 'data/train.csv'
IMG_DIR    = 'data/train_images'
IMG_SIZE   = 224
BATCH_SIZE = 32
NUM_EPOCHS = 30
LR         = 1e-4
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.1
SEED       = 42
# ───────────────────────────────────────────────────────────────────────────────

import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

## 1. Data Loading with Improved Augmentation

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
from dataset import APTOSDataset, get_improved_transforms, compute_class_weights

os.makedirs('results', exist_ok=True)

df = pd.read_csv(CSV_PATH)

train_df, temp_df = train_test_split(
    df, test_size=0.30, stratify=df['diagnosis'], random_state=SEED
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df['diagnosis'], random_state=SEED
)

print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')
print(f'Train class distribution:\n{train_df["diagnosis"].value_counts().sort_index()}\n')

train_transform, val_transform = get_improved_transforms(IMG_SIZE)

import platform
num_workers = 0 if platform.system() == 'Darwin' else 2
pin_memory  = platform.system() != 'Darwin'

train_loader = DataLoader(APTOSDataset(train_df, IMG_DIR, train_transform),
                          batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=num_workers, pin_memory=pin_memory)
val_loader   = DataLoader(APTOSDataset(val_df,   IMG_DIR, val_transform),
                          batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=num_workers, pin_memory=pin_memory)
test_loader  = DataLoader(APTOSDataset(test_df,  IMG_DIR, val_transform),
                          batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=num_workers, pin_memory=pin_memory)

## 2. Class Weights

In [ ]:
class_weights = compute_class_weights(train_df, num_classes=5).to(DEVICE)
print('Class weights (inverse frequency):')
CLASS_NAMES = ['No DR (0)', 'Mild (1)', 'Moderate (2)', 'Severe (3)', 'Proliferative (4)']
for name, w in zip(CLASS_NAMES, class_weights.cpu()):
    print(f'  {name}: {w:.3f}')

## 3. EfficientNet-B0 Model

In [ ]:
from improved_model import ImprovedDRClassifier
import torch.nn as nn
import torch.optim as optim

model = ImprovedDRClassifier(num_classes=5).to(DEVICE)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters:     {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')

## 4. Training

- **Loss**: CrossEntropyLoss with class weights + label smoothing 0.1
- **Optimizer**: AdamW (weight decay prevents overfitting on small dataset)
- **Schedule**: CosineAnnealingLR (smooth LR decay, no manual plateau detection)
- **Checkpoint**: best val macro F1 (appropriate for imbalanced 5-class task)

In [ ]:
from train import train_improved, evaluate, print_metrics

criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=LABEL_SMOOTHING)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

history = train_improved(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    criterion=criterion,
    scheduler=scheduler,
    device=DEVICE,
    num_epochs=NUM_EPOCHS,
)

torch.save(model.state_dict(), 'results/improved_efficientnet.pth')
print('Model saved to results/improved_efficientnet.pth')

## 5. Training Curves

In [ ]:
epochs = range(1, NUM_EPOCHS + 1)

fig, axes = plt.subplots(1, 3, figsize=(17, 4))

axes[0].plot(epochs, history['train_loss'], label='Train', marker='o', ms=3)
axes[0].plot(epochs, history['val_loss'],   label='Val',   marker='s', ms=3)
axes[0].set_title('Loss — Improved Model')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(epochs, history['train_acc'], label='Train', marker='o', ms=3)
axes[1].plot(epochs, history['val_acc'],   label='Val',   marker='s', ms=3)
axes[1].set_title('Accuracy — Improved Model')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].plot(epochs, history['val_f1'], label='Val Macro F1', marker='^', ms=3, color='darkorange')
axes[2].set_title('Macro F1 — Improved Model')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Macro F1')
axes[2].legend(); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('results/improved_training_curves.png', bbox_inches='tight')
plt.show()

## 6. Evaluation on Test Set

In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix

test_loss, test_acc, test_labels, test_preds = evaluate(model, test_loader, criterion, DEVICE)
print(f'Test Loss: {test_loss:.4f} | Test Accuracy: {test_acc:.4f}')
cm = print_metrics(test_labels, test_preds)

fig, ax = plt.subplots(figsize=(7, 6))
short_names = ['No DR\n(0)', 'Mild\n(1)', 'Moderate\n(2)', 'Severe\n(3)', 'Proliferative\n(4)']
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=short_names, yticklabels=short_names, ax=ax)
ax.set_title('Confusion Matrix — Improved Model (Test Set)', fontsize=13)
ax.set_ylabel('True Label')
ax.set_xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('results/improved_confusion_matrix.png', bbox_inches='tight')
plt.show()

## 7. Grad-CAM

In [ ]:
from gradcam import visualize_gradcam

test_dataset = APTOSDataset(test_df, IMG_DIR, transform=val_transform)
fig = visualize_gradcam(model, test_dataset, DEVICE, num_samples=8, seed=SEED)
fig.savefig('results/improved_gradcam.png', bbox_inches='tight')
plt.show()